In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import SpaGCN as spg
from PIL import Image
import torch
import random
import os
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import scipy.sparse as sp

/home/ec2-user/miniforge3/envs/spatial/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/home/ec2-user/miniforge3/envs/spatial/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/home/ec2-user/miniforge3/envs/spatial/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_mtx from `anndata` is deprecated. Import anndata.io.read_mtx instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)


In [2]:
data_dir = os.path.expanduser("../data/151673")
table_dir = os.path.expanduser("../results/tables")
figure_dir = os.path.expanduser("../results/figures") 

In [3]:
if not hasattr(sp.spmatrix, "A"):
    sp.spmatrix.A = property(lambda self: self.toarray())

In [4]:
adata = sc.read_h5ad(os.path.expanduser(
    "../data/processed/02_preprocessed.h5ad"))

In [5]:
img = np.array(Image.open(os.path.join(data_dir, "spatial", "151673_full_image.tif")))
print(f"img shape: {img.shape}")

/home/ec2-user/miniforge3/envs/spatial/lib/python3.12/site-packages/PIL/Image.py:3574: DecompressionBombWarning: Image size (177742224 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


img shape: (13332, 13332, 3)


In [6]:
x_pixel = adata.obsm["spatial"][:, 0].tolist()
y_pixel = adata.obsm["spatial"][:, 1].tolist()


if "array_row" in adata.obs.columns and "array_col" in adata.obs.columns:
    x_array = adata.obs["array_row"].tolist()
    y_array = adata.obs["array_col"].tolist()
else:
    x_array = x_pixel
    y_array = y_pixel

In [7]:
# r_seed = t_seed = n_seed = 100

# adata.obs["spagcn_pred"] = spg.detect_spatial_domains_ez_mode(
#     adata, img,
#     x_array, y_array, x_pixel, y_pixel,
#     n_clusters=7,
#     histology=True,
#     s=1, b=49, p=0.5,
#     r_seed=r_seed, t_seed=t_seed, n_seed=n_seed
# )
# adata.obs["spagcn_pred"] = adata.obs["spagcn_pred"].astype("category")

In [8]:
adj = spg.calculate_adj_matrix(
    x=x_pixel, y=y_pixel,
    x_pixel=x_pixel, y_pixel=y_pixel,
    image=img, beta=49, alpha=1, histology=True
)

adata_spg = adata.copy()
spg.prefilter_genes(adata_spg, min_cells=3)
spg.prefilter_specialgenes(adata_spg)
sc.pp.normalize_per_cell(adata_spg)
sc.pp.log1p(adata_spg)

Calculateing adj matrix using histology image...
Var of c0,c1,c2 =  91.80770523260091 566.0463832114333 70.16888798940096
Var of x,y,z =  4415447.918876557 5514627.518783892 5514627.518783891


/tmp/ipykernel_10376/1113132380.py:10: FutureWarning: Use `sc.pp.normalize_total` instead.
  sc.pp.normalize_per_cell(adata_spg)


In [9]:
p = 0.5
l = spg.search_l(p, adj, start=0.01, end=1000, tol=0.01, max_run=100)

Run 1: l [0.01, 1000], p [0.0, 204.32283]
Run 2: l [0.01, 500.005], p [0.0, 45.192196]
Run 3: l [0.01, 250.0075], p [0.0, 8.11254]
Run 4: l [0.01, 125.00874999999999], p [0.0, 1.1414242]
Run 5: l [62.509375, 125.00874999999999], p [0.069051385, 1.1414242]
Run 6: l [93.7590625, 125.00874999999999], p [0.43540514, 1.1414242]
Run 7: l [93.7590625, 109.38390625], p [0.43540514, 0.7434268]
Run 8: l [93.7590625, 101.571484375], p [0.43540514, 0.5785266]
recommended l =  97.66527343749999


In [10]:
n_clusters = 7
r_seed = t_seed = n_seed = 100
res = spg.search_res(adata_spg, adj, l, n_clusters,
                     start=0.7, step=0.1, tol=5e-3,
                     lr=0.05, max_epochs=20,
                     r_seed=r_seed, t_seed=t_seed, n_seed=n_seed)

print(f"res = {res}")

Start at res =  0.7 step =  0.1
Initializing cluster centers with louvain, resolution =  0.7


/home/ec2-user/miniforge3/envs/spatial/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ec2-user/miniforge3/envs/spatial/lib/python3.12/site-packages/SpaGCN/models.py:69: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
  sc.tl.leiden(adata,resolution=res)


Epoch  0


/home/ec2-user/miniforge3/envs/spatial/lib/python3.12/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch  10
Res =  0.7 Num of clusters =  8
Initializing cluster centers with louvain, resolution =  0.6


/home/ec2-user/miniforge3/envs/spatial/lib/python3.12/site-packages/SpaGCN/models.py:69: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
  sc.tl.leiden(adata,resolution=res)


Epoch  0
Epoch  10
Res =  0.6 Num of clusters =  6
Step changed to 0.05
Initializing cluster centers with louvain, resolution =  0.6499999999999999


/home/ec2-user/miniforge3/envs/spatial/lib/python3.12/site-packages/SpaGCN/models.py:69: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
  sc.tl.leiden(adata,resolution=res)


Epoch  0
Epoch  10
Res =  0.6499999999999999 Num of clusters =  7
recommended res =  0.6499999999999999
res = 0.6499999999999999


In [11]:
clf = spg.SpaGCN()
clf.set_l(l)
random.seed(r_seed)
torch.manual_seed(t_seed)
np.random.seed(n_seed)
clf.train(
    adata_spg, adj, init_spa=True, init="louvain",
    res=res, tol=5e-3, lr=0.05, max_epochs=200
)

Initializing cluster centers with louvain, resolution =  0.6499999999999999


/home/ec2-user/miniforge3/envs/spatial/lib/python3.12/site-packages/SpaGCN/models.py:69: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
  sc.tl.leiden(adata,resolution=res)


Epoch  0
Epoch  10
Epoch  20
Epoch  30
Epoch  40
Epoch  50
Epoch  60
Epoch  70
Epoch  80
delta_label  0.0049986113 < tol  0.005
Reach tolerance threshold. Stopping training.
Total epoch: 82


In [12]:
y_pred, prob = clf.predict()
adata.obs["spagcn_pred"] = y_pred
adata.obs["spagcn_pred"] = adata.obs["spagcn_pred"].astype("category")

In [13]:
adata.obs["spagcn_refined"] = spg.spatial_domains_refinement_ez_mode(
    sample_id=adata.obs.index.tolist(),
    pred=adata.obs["spagcn_pred"].tolist(),
    x_array=x_array, y_array=y_array,
    shape="hexagon"
)
adata.obs["spagcn_refined"] = adata.obs["spagcn_refined"].astype("category")

Calculateing adj matrix using xy only...


In [14]:
mask = adata.obs["ground_truth"].notna()
ari = adjusted_rand_score(adata.obs.loc[mask, "ground_truth"], adata.obs.loc[mask, "spagcn_refined"])
nmi = normalized_mutual_info_score(adata.obs.loc[mask, "ground_truth"], adata.obs.loc[mask, "spagcn_refined"])
print(f"SpaGCN - ARI: {ari:.4f}, NMI: {nmi:.4f}")

SpaGCN - ARI: 0.3163, NMI: 0.4593


In [15]:
adata.write(os.path.join("../data/processed/04_spagcn.h5ad"))

In [16]:
adata.obs[["ground_truth", "leiden", "spagcn_refined"]].to_csv(
    os.path.join(table_dir, "clustering_results_python.csv"))